# 02 · pandas wrangling

**Session 4 — Wrangle and Version** · about 40 minutes

**You will learn to**
- summarise groups with `groupby`, named aggregation and `transform`
- combine tables with `merge` safely: matching keys, `validate=` and row counts
- reshape tables with `pivot_table` and `melt`
- parse numbers written the Catalan/Spanish way (`1.234,5`)

**Data:** IMDB top 1000 (Kaggle); electricity consumption by municipality and sector (Transparència Catalunya); Catalan municipalities (ICGC). See `data/README.md`.

**Outline**
0. Before you start (no AI)
1. Split–apply–combine
2. Hierarchical data
3. Combining tables safely
4. Reshaping
5. Numbers with commas
6. Your dataset

> **AI guardrails** — *Think first · Own it · Verify · Disclose · Protect.* AI tools are allowed in work time, but answer the **No-AI** questions on paper first, and make every ✅ self-check pass with code you can explain line by line.

## 0. Before you start (no AI)

A table has 1,000 films by 548 directors.

1. How many rows does `df.groupby("Director")["IMDB_Rating"].mean()` have?
2. How many rows does `df.groupby("Director")["IMDB_Rating"].transform("mean")` have?
3. You merge 1,000 films with a table of directors' nationalities and get 1,040 rows. What happened?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
from pathlib import Path

DATA = Path("../data")
OUTPUTS = Path("../outputs")
OUTPUTS.mkdir(exist_ok=True)
assert DATA.exists(), "Start Jupyter from the repository folder: notebooks/ must sit next to data/."

In [ ]:
imdb = pd.read_csv(DATA / "imdb/imdb_top_1000.csv")
imdb["gross_usd"] = pd.to_numeric(imdb["Gross"].str.replace(",", ""), errors="coerce")
imdb["Director"].nunique()

## 1. Split–apply–combine

`groupby` splits the table into groups, applies a function to each group and combines the results.

- `agg` returns **one row per group**;
- `transform` returns **one value per original row** (same shape as the input).

In [ ]:
per_director = (
    imdb.groupby("Director")
    .agg(mean_rating=("IMDB_Rating", "mean"),
         n_films=("Series_Title", "count"),
         total_gross_musd=("gross_usd", lambda s: s.sum() / 1e6))
    .sort_values("n_films", ascending=False)
)
per_director.head()

In [ ]:
imdb["director_mean"] = imdb.groupby("Director")["IMDB_Rating"].transform("mean")
imdb[["Series_Title", "Director", "IMDB_Rating", "director_mean"]].head()

### TASK I — above their director's average

1. Add a column `vs_director` = film rating − its director's mean rating.
2. Among directors with **at least 5 films**, build `consistent`: one row per director with
   `n_films` and `spread` (= max − min rating), sorted by `spread` (most consistent first).

*Predict first:* will `vs_director` have the same number of rows as `imdb`?

In [ ]:
# TASK I — your code here
consistent = None

In [ ]:
# ✅ Self-check TASK I
assert "vs_director" in imdb and len(imdb["vs_director"]) == len(imdb)
assert abs(imdb["vs_director"].mean()) < 1e-9, "Deviations from a group mean average to zero."
assert consistent is not None and len(consistent) == 35, "35 directors have at least 5 films."
assert consistent["spread"].is_monotonic_increasing
print("✅ TASK I passed")

## 2. Hierarchical data (MultiIndex)

Several index levels (here city and year) are common in panel data.

In [ ]:
index = pd.MultiIndex.from_product([["Barcelona", "Madrid", "Valencia"], [2024, 2025]],
                                   names=["city", "year"])
panel = pd.DataFrame({"population_m": [1.66, 1.70, 3.33, 3.42, 0.81, 0.83],
                      "unemployment_pct": [8.5, 8.3, 10.2, 9.8, 11.1, 10.9]}, index=index)
panel

In [ ]:
print(panel.loc["Barcelona"])
print(panel.xs(2025, level="year"))
panel.groupby(level="city").agg(pop_growth=("population_m", lambda s: s.iloc[-1] - s.iloc[0]),
                                mean_unemployment=("unemployment_pct", "mean"))

*Toy numbers for illustration.*

## 3. Combining tables safely

Electricity consumption per municipality and sector (2013–2024) and the ICGC table of municipalities
(with their area). Goal: consumption **per km²**.

In [ ]:
import geopandas as gpd

energy = pd.read_parquet(DATA / "examples/energy_consumption_municipalities.parquet")
municipis = gpd.read_file(DATA / "icgc/divisions-administratives-v2r1-municipis-1000000-20250730.json")
print(energy[["cdmun", "municipi"]].head(3))
print(municipis[["CODIMUNI", "NOMMUNI", "AREAM5000"]].head(3))

The keys do not match yet: `cdmun` is the integer `8001`, `CODIMUNI` is the text `"080018"`.
Both are official municipality codes: INE's 5 digits (`08001`, the leading zero lost when the
column became a number) and the same code plus a **check digit** (`080018`).

### Verify the AI

A tempting shortcut is to merge on the municipality **name**. Count what that loses:

In [ ]:
by_name = energy.merge(municipis, left_on="municipi", right_on="NOMMUNI", how="left")
print(f"rows without a match when joining by name: {by_name['NOMMUNI'].isna().sum():,} of {len(by_name):,}")
by_name.loc[by_name["NOMMUNI"].isna(), "municipi"].drop_duplicates().head()

Upper case, accents, articles (*L'Hospitalet*…): names are not keys. Build the same **code** on both sides:

In [ ]:
energy["ine5"] = energy["cdmun"].astype(str).str.zfill(5)
municipis["ine5"] = municipis["CODIMUNI"].str[:5]

year = energy[energy["any"] == 2023]
merged = year.merge(municipis[["ine5", "NOMMUNI", "NOMCOMAR", "AREAM5000"]],
                    on="ine5", how="left", validate="many_to_one")
print(len(year), len(merged), "rows without area:", merged["AREAM5000"].isna().sum())

- `validate="many_to_one"` raises an error if a municipality appears twice on the right
  (which would silently duplicate rows).
- Comparing row counts before and after, and counting missing values, catches lost matches.
  Make both checks a habit, especially when the merge code came from an AI assistant.

### TASK II — energy density

With `merged` (year 2023):

1. Compute total consumption per municipality (all sectors) in GWh, then `gwh_per_km2`.
2. Build `top10`: the 10 municipalities with the highest `gwh_per_km2`, with columns
   `NOMMUNI`, `NOMCOMAR`, `gwh`, `gwh_per_km2`.

In [ ]:
# TASK II — your code here
top10 = None

In [ ]:
# ✅ Self-check TASK II
assert top10 is not None and len(top10) == 10
assert list(top10.columns) == ["NOMMUNI", "NOMCOMAR", "gwh", "gwh_per_km2"]
assert top10["gwh_per_km2"].is_monotonic_decreasing
assert "Barcelona" in set(top10["NOMMUNI"]), "Dense cities should appear near the top."
print("✅ TASK II passed")

`pd.concat` stacks tables with the same columns (e.g. one file per year):

In [ ]:
y2013, y2024 = energy[energy["any"] == 2013], energy[energy["any"] == 2024]
both = pd.concat([y2013, y2024], ignore_index=True)
len(y2013), len(y2024), len(both)

## 4. Reshaping: `pivot_table` and `melt`

*Long* tables (one row per municipality–year–sector) are best for computing; *wide* tables
(one column per sector) are often best for reading and for some plots.

In [ ]:
wide = energy.pivot_table(index="any", columns="descripcio_sector", values="consum_kwh",
                          aggfunc="sum") / 1e9
wide.round(2)

In [ ]:
ax = wide.plot(figsize=(8, 4))
ax.set_ylabel("Electricity consumption (TWh)")
ax.set_xlabel("Year")
ax.set_title("Electricity consumption in Catalonia by sector")
ax.legend(title=None, fontsize=8, ncols=2)
plt.show()

### TASK III — back to long

Turn `wide` back into a long table `long` with columns `any`, `descripcio_sector`, `twh`
using `reset_index()` and `melt`.

In [ ]:
# TASK III — your code here
long = None

In [ ]:
# ✅ Self-check TASK III
assert long is not None and list(long.columns) == ["any", "descripcio_sector", "twh"]
assert len(long) == wide.shape[0] * wide.shape[1]
assert abs(long["twh"].sum() - wide.to_numpy().sum()) < 1e-9
print("✅ TASK III passed")

## 5. Numbers written with commas

Catalan and Spanish sources write `1.234,5` (dot for thousands, comma for decimals).
When pandas reads them as text, convert them explicitly.

In [ ]:
raw = pd.DataFrame({"value": ["1,00", "2,50", "3,750", "1.234,5"]})
raw["value_num"] = pd.to_numeric(raw["value"].str.replace(".", "", regex=False)
                                             .str.replace(",", ".", regex=False))
raw

Better: tell `read_csv` how the file is written.

In [ ]:
from io import StringIO

csv_text = "municipi;consum\nGirona;1.234,5\nLleida;987,25\n"
pd.read_csv(StringIO(csv_text), sep=";", decimal=",", thousands=".")

## 6. Your dataset

In your group repository: one `groupby` summary of your data, and — if you combine two sources —
a merge on a **code**, with `validate=` and a row-count check printed below it.

**Next:** `03_open_data_apis.ipynb`.